In [ ]:
import importlib
import sys

# Force reload of helpers module to pick up latest changes
if 'helpers' in sys.modules:
    importlib.reload(sys.modules['helpers'])
    importlib.reload(sys.modules['helpers.database'])
    importlib.reload(sys.modules['helpers.logging_config'])

In [0]:
# ==============================================================================
# CONFIGURATION & HELPER FUNCTIONS
# ==============================================================================
from helpers import (
    create_connection, write_gold_table, setup_logger,
    transform_customer_full_pipeline, transform_staff_full_pipeline,
    transform_store_full_pipeline, transform_car_full_pipeline,
    generate_date_dimension, generate_role_playing_date_dimensions,
    build_staff_hierarchy_bridge, transform_equipment_dimension,
    build_equipment_bridges,
    BronzeLoader
)
import time
from datetime import date

# Setup logging
logger = setup_logger("load_dimensions")

# Initialize connection
c = create_connection(spark, dbutils)
logger.info("=" * 70)
logger.info("DIMENSION LOAD JOB STARTED")
logger.info("=" * 70)

WATERMARK_COLUMN = "last_update"
bronze_loader = BronzeLoader(spark, dbutils)


def load_and_persist_bronze(table_name: str):
    """Full-load raw table into bronze and keep watermarks in sync."""
    return bronze_loader.load_full_to_bronze(
        table_name=table_name,
        watermark_column=WATERMARK_COLUMN,
        updated_by="load_all_dim",
    )


In [0]:
# ==============================================================================
# BRONZE LAYER: Load all source tables (persisted to bronze)
# ==============================================================================

inventory_bronze = load_and_persist_bronze("inventory")
car_bronze = load_and_persist_bronze("car")
inventory_equipment_bronze = load_and_persist_bronze("inventory_equipment")
equipment_bronze = load_and_persist_bronze("equipment")
staff_bronze = load_and_persist_bronze("staff")
city_bronze = load_and_persist_bronze("city")
address_bronze = load_and_persist_bronze("address")
store_bronze = load_and_persist_bronze("store")
country_bronze = load_and_persist_bronze("country")
customer_bronze = load_and_persist_bronze("customer")


In [0]:
# ==============================================================================
# GOLD: DIM_DATE
# ==============================================================================

logger.info("GOLD: Building dim_date")
start_time = time.time()

START_DATE = date(2000, 1, 1)
END_DATE = date(2027, 12, 31)

# Generate base date dimension
dim_date = generate_date_dimension(
    spark,
    start_date=START_DATE,
    end_date=END_DATE
)

write_gold_table(dim_date, "dim_date")
logger.info(f"GOLD: dim_date completed in {time.time() - start_time:.2f}s")


In [ ]:
# ==============================================================================
# GOLD: DATE DIMENSION COPIES (Role-Playing Dimensions)
# ==============================================================================

logger.info("GOLD: Building role-playing date dimensions")
start_time = time.time()

# Define role-playing dimension configurations
role_playing_dates = [
    ("dim_service_date", "service_date_key"),
    ("dim_rental_date", "rental_date_key"),
    ("dim_return_date", "return_date_key"),
    ("dim_payment_date", "payment_date_key"),
    ("dim_payment_deadline_date", "payment_deadline_date_key")
]

# Generate all role-playing dimensions
role_playing_dims = generate_role_playing_date_dimensions(dim_date, role_playing_dates)

# Write each role-playing dimension
for table_name, df in role_playing_dims:
    write_gold_table(df, table_name)

logger.info(f"GOLD: All {len(role_playing_dates)} role-playing date dimensions completed in {time.time() - start_time:.2f}s")

In [0]:
# ==============================================================================
# GOLD: DIM_STAFF + BRIDGE_STAFF_HIERARCHY
# ==============================================================================

logger.info("GOLD: Building dim_staff and staff hierarchy")
start_time = time.time()

# Build staff hierarchy bridge using helper
bridge_staff_hierarchy = build_staff_hierarchy_bridge(staff_bronze, max_depth=10)
write_gold_table(bridge_staff_hierarchy, "dim_staff_hierarchy")

# Build dim_staff
dim_staff = transform_staff_full_pipeline(
    staff_bronze, address_bronze, city_bronze, country_bronze
)
write_gold_table(dim_staff, "dim_staff")

logger.info(f"GOLD: dim_staff and hierarchy completed in {time.time() - start_time:.2f}s")


In [0]:
# ==============================================================================
# GOLD: DIM_STORE (SCD TYPE 2 Ready)
# ==============================================================================

logger.info("GOLD: Building dim_store")
start_time = time.time()

dim_store = transform_store_full_pipeline(
    store_bronze, staff_bronze, address_bronze, city_bronze, country_bronze
)

write_gold_table(dim_store, "dim_store", mode="overwrite")

logger.info(f"GOLD: dim_store completed in {time.time() - start_time:.2f}s")

In [0]:
# ==============================================================================
# GOLD: DIM_CAR + DIM_EQUIPMENT + EQUIPMENT BRIDGES
# ==============================================================================

logger.info("GOLD: Building dim_car, dim_equipment, and equipment bridges")
start_time = time.time()

# Build dim_equipment
dim_equipment = transform_equipment_dimension(equipment_bronze)
write_gold_table(dim_equipment, "dim_equipment")

# Build equipment bridges
bridge_equipment_group_equipment, bridge_car_equipment = build_equipment_bridges(
    inventory_equipment_bronze
)
write_gold_table(bridge_equipment_group_equipment, "bridge_equipment_group_equipment")
write_gold_table(bridge_car_equipment, "bridge_car_equipment")

# Build dim_car
dim_car = transform_car_full_pipeline(
    inventory_bronze, car_bronze, inventory_equipment_bronze, equipment_bronze
)
write_gold_table(dim_car, "dim_car")

logger.info(f"GOLD: dim_car and equipment tables completed in {time.time() - start_time:.2f}s")


In [0]:
# ==============================================================================
# GOLD: DIM_CUSTOMER
# ==============================================================================

logger.info("GOLD: Building dim_customer")
start_time = time.time()

dim_customer = transform_customer_full_pipeline(
    customer_bronze, address_bronze, city_bronze, country_bronze
)

write_gold_table(dim_customer, "dim_customer", partition_by=["customer_country"])

logger.info(f"GOLD: dim_customer completed in {time.time() - start_time:.2f}s")

# ==============================================================================
# JOB COMPLETION
# ==============================================================================
logger.info("=" * 70)
logger.info("DIMENSION LOAD JOB COMPLETED SUCCESSFULLY")
logger.info("=" * 70)
